# 🧹 Kompas Data Cleaning & Sampling

**Goal:** Prepare Kompas data for IndoBERT sentiment analysis + BiLSTM time series  
**Input:** kompas_articles.csv (raw)  
**Output:**
- Cleaned full dataset
- Random seed data (600-1000) for manual labeling (seed=42)
- Excel template for labeling

---

## 📦 Step 1: Setup

In [1]:
import pandas as pd
import numpy as np
import re
from datetime import datetime
import os

# Set random seed for reproducibility
np.random.seed(42)

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 100)

print("✅ Libraries loaded!")
print(f"Random seed: 42 (reproducible sampling)")

✅ Libraries loaded!
Random seed: 42 (reproducible sampling)


## 📥 Step 2: Load Data

In [5]:
print("="*70)
print("📥 LOADING DATA")
print("="*70)

# Load Kompas data
df = pd.read_csv('../data/data_berita/fixed/kompas_articles.csv', encoding='utf-8')

print(f"\n✅ Loaded: {len(df):,} articles")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nFirst rows:")
display(df.head(3))

📥 LOADING DATA

✅ Loaded: 9,980 articles

Columns: ['title', 'tanggal', 'content']

First rows:


,title,tanggal,content
0,Menko Polkam Pastikan Perayaan Malam Tahun Baru 2026 di Indonesia Berjalan Kondusif,31 Desember 2025,"JAKARTA, KOMPAS.com -Menteri Koordinator Bidang Politik dan Keamanan (Menko Polkam) Djamari Chan..."
1,"Indonesia 2025: Potret Serdadu dan Polisi Maju, Sipil Mundur",31 Desember 2025,"Di BALIKekonomi yang tidak bisa disebut baik-baik saja, lapangan kerja sulit, pemutusan hubungan..."
2,Bagaimana Bisa 68 Anak Indonesia Terpapar White Supremacy Neo-Nazi?,31 Desember 2025,"JAKARTA, KOMPAS.com- Polisi mengungkap ada puluhan anak Indonesia terpapar ideologi neo-Nazi dan..."


## 📅 Step 3: Date Parsing & Filtering

In [6]:
print("="*70)
print("📅 DATE PARSING & FILTERING")
print("="*70)

def parse_kompas_date(date_str):
    """
    Parse Kompas Indonesian date format: 'DD Month YYYY'
    Example: '31 Desember 2025' -> datetime(2025, 12, 31)
    """
    try:
        # Map Indonesian months to numbers
        months = {
            'Januari': '01', 'Februari': '02', 'Maret': '03', 'April': '04',
            'Mei': '05', 'Juni': '06', 'Juli': '07', 'Agustus': '08',
            'September': '09', 'Oktober': '10', 'November': '11', 'Desember': '12'
        }
        
        parts = str(date_str).strip().split()
        if len(parts) == 3:
            day, month_name, year = parts
            month = months.get(month_name)
            if month:
                date_str_eng = f"{year}-{month}-{day.zfill(2)}"
                return pd.to_datetime(date_str_eng, format='%Y-%m-%d')
    except Exception as e:
        pass
    return None

# Parse dates
print("\nParsing Indonesian dates...")
df['date'] = df['tanggal'].apply(parse_kompas_date)

# Remove rows with invalid dates
before = len(df)
df = df.dropna(subset=['date'])
print(f"   Parsed: {len(df)}/{before} (removed {before - len(df)} invalid)")

# Filter 2019-2024 only
start_date = pd.to_datetime('2019-09-01')
end_date = pd.to_datetime('2024-09-30')

before = len(df)
df = df[(df['date'] >= start_date) & (df['date'] <= end_date)]
print(f"\nFiltered to 2019-09-01 to 2024-09-30:")
print(f"   Removed: {before - len(df)} articles outside range")
print(f"   Remaining: {len(df):,} articles")

if len(df) == 0:
    print(f"\n⚠️ WARNING: No articles in 2019-2024 range!")
    print(f"   Most data is from 2025-2026")
    print(f"   You may need to re-scrape with correct date filters")
else:
    print(f"\n📊 Date range:")
    print(f"   Min: {df['date'].min().date()}")
    print(f"   Max: {df['date'].max().date()}")

📅 DATE PARSING & FILTERING

Parsing Indonesian dates...
   Parsed: 9980/9980 (removed 0 invalid)

Filtered to 2019-09-01 to 2024-09-30:
   Removed: 2825 articles outside range
   Remaining: 7,155 articles

📊 Date range:
   Min: 2022-01-20
   Max: 2024-09-30


## 🧹 Step 4: Text Cleaning

In [7]:
print("="*70)
print("🧹 TEXT CLEANING")
print("="*70)

def clean_text(text):
    """
    Clean text for sentiment analysis:
    - Remove URLs
    - Remove emails
    - Remove extra whitespace
    - Remove special characters except punctuation
    """
    if pd.isna(text) or text == "N/A":
        return ""
    
    text = str(text)
    
    # Remove URLs
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)
    
    # Remove emails
    text = re.sub(r'\S+@\S+', '', text)
    
    # Remove "Baca juga:" references (Kompas specific)
    text = re.sub(r'Baca juga:[^\n]*', '', text)
    
    # Remove newlines and tabs
    text = text.replace('\n', ' ').replace('\r', ' ').replace('\t', ' ')
    
    # Remove extra whitespace
    text = ' '.join(text.split())
    
    return text

# Apply cleaning to title and content
print("\nCleaning titles...")
df['title_clean'] = df['title'].apply(clean_text)

print("Cleaning content...")
df['content_clean'] = df['content'].apply(clean_text)

# Remove rows with empty content after cleaning
before = len(df)
df = df[df['content_clean'].str.len() > 50]  # Minimum 50 chars
print(f"\n✅ Cleaned!")
print(f"   Removed {before - len(df)} articles with insufficient content")
print(f"   Remaining: {len(df):,} articles")

if len(df) > 0:
    # Show sample
    print(f"\n📋 Sample cleaned text:")
    sample = df.iloc[0]
    print(f"\nOriginal title: {sample['title'][:100]}")
    print(f"Cleaned title: {sample['title_clean'][:100]}")
    print(f"\nOriginal content (first 200 chars): {sample['content'][:200]}")
    print(f"Cleaned content (first 200 chars): {sample['content_clean'][:200]}")

🧹 TEXT CLEANING

Cleaning titles...
Cleaning content...

✅ Cleaned!
   Removed 0 articles with insufficient content
   Remaining: 7,155 articles

📋 Sample cleaned text:

Original title: Pilkada 2024: Katalis Integrasi Pembangunan Daerah dan Agenda Politik Nasional
Cleaned title: Pilkada 2024: Katalis Integrasi Pembangunan Daerah dan Agenda Politik Nasional

Original content (first 200 chars): DI TENGAHkegaduhan politik yang tak pernah surut, Pemilihan Kepala Daerah (Pilkada) serentak 2024 mendatang, memberikan peluang unik bagi Indonesia. Bukan sekadar rutinitas pemilihan, melainkan moment
Cleaned content (first 200 chars): DI TENGAHkegaduhan politik yang tak pernah surut, Pemilihan Kepala Daerah (Pilkada) serentak 2024 mendatang, memberikan peluang unik bagi Indonesia. Bukan sekadar rutinitas pemilihan, melainkan moment


## 🔄 Step 5: Remove Duplicates

In [8]:
print("="*70)
print("🔄 REMOVING DUPLICATES")
print("="*70)

if len(df) == 0:
    print("\n⚠️ No data to deduplicate (empty dataset)")
else:
    before = len(df)
    
    # Remove exact title duplicates
    df = df.drop_duplicates(subset=['title_clean'], keep='first')
    after_title = len(df)
    print(f"\n1. By title: Removed {before - after_title} duplicates")
    
    # Remove similar content (first 100 chars)
    df['content_hash'] = df['content_clean'].str[:100]
    df = df.drop_duplicates(subset=['content_hash'], keep='first')
    after_content = len(df)
    print(f"2. By content: Removed {after_title - after_content} duplicates")
    
    df = df.drop(columns=['content_hash'])
    
    print(f"\n✅ Total removed: {before - after_content} duplicates")
    print(f"   Final dataset: {len(df):,} unique articles")

🔄 REMOVING DUPLICATES

1. By title: Removed 4 duplicates
2. By content: Removed 317 duplicates

✅ Total removed: 321 duplicates
   Final dataset: 6,834 unique articles


## 📊 Step 6: Final Dataset Statistics

In [9]:
print("="*70)
print("📊 FINAL DATASET STATISTICS")
print("="*70)

if len(df) == 0:
    print("\n❌ Empty dataset after filtering!")
    print("\n⚠️ ACTION REQUIRED:")
    print("   Most Kompas data is from 2025-2026")
    print("   Need to re-scrape with date filters: 2019-2024")
else:
    print(f"\n✅ Total articles: {len(df):,}")
    print(f"\n📅 Date range:")
    print(f"   From: {df['date'].min().date()}")
    print(f"   To: {df['date'].max().date()}")
    print(f"   Days: {(df['date'].max() - df['date'].min()).days}")
    
    # Monthly distribution
    df['year_month'] = df['date'].dt.to_period('M')
    monthly = df['year_month'].value_counts().sort_index()
    
    print(f"\n📈 Monthly distribution:")
    print(f"   Mean: {monthly.mean():.1f} articles/month")
    print(f"   Median: {monthly.median():.0f} articles/month")
    print(f"   Min: {monthly.min()} articles/month")
    print(f"   Max: {monthly.max()} articles/month")
    
    if len(monthly) > 0:
        print(f"\n📋 First 6 months:")
        for period, count in monthly.head(6).items():
            print(f"   {period}: {count:4d} articles")
        
        print(f"\n📋 Last 6 months:")
        for period, count in monthly.tail(6).items():
            print(f"   {period}: {count:4d} articles")
    
    # Text length stats
    df['content_length'] = df['content_clean'].str.len()
    print(f"\n📏 Content length:")
    print(f"   Mean: {df['content_length'].mean():.0f} chars")
    print(f"   Median: {df['content_length'].median():.0f} chars")
    print(f"   Min: {df['content_length'].min():.0f} chars")
    print(f"   Max: {df['content_length'].max():.0f} chars")

📊 FINAL DATASET STATISTICS

✅ Total articles: 6,834

📅 Date range:
   From: 2022-01-20
   To: 2024-09-30
   Days: 984

📈 Monthly distribution:
   Mean: 207.1 articles/month
   Median: 202 articles/month
   Min: 95 articles/month
   Max: 366 articles/month

📋 First 6 months:
   2022-01:   95 articles
   2022-02:  223 articles
   2022-03:  226 articles
   2022-04:  187 articles
   2022-05:  262 articles
   2022-06:  233 articles

📋 Last 6 months:
   2024-04:  133 articles
   2024-05:  180 articles
   2024-06:  198 articles
   2024-07:  164 articles
   2024-08:  202 articles
   2024-09:  260 articles

📏 Content length:
   Mean: 794 chars
   Median: 652 chars
   Min: 137 chars
   Max: 6132 chars


## 💾 Step 7: Save Cleaned Full Dataset

In [10]:
print("="*70)
print("💾 SAVING CLEANED FULL DATASET")
print("="*70)

if len(df) == 0:
    print("\n⚠️ Skipping save (empty dataset)")
else:
    # Select essential columns for sentiment analysis
    df_clean = df[[
        'date',
        'title_clean',
        'content_clean',
        'content_length'
    ]].copy()
    
    # Rename for clarity
    df_clean.columns = ['date', 'title', 'content', 'content_length']
    
    # Sort by date
    df_clean = df_clean.sort_values('date').reset_index(drop=True)
    
    # Save
    output_file = 'kompas_cleaned_full.csv'
    df_clean.to_csv(output_file, index=False, encoding='utf-8-sig')
    
    file_size = os.path.getsize(output_file) / (1024**2)
    
    print(f"\n✅ Saved: {output_file}")
    print(f"   Rows: {len(df_clean):,}")
    print(f"   Columns: {df_clean.columns.tolist()}")
    print(f"   Size: {file_size:.2f} MB")
    
    print(f"\n📋 Sample:")
    display(df_clean.head(5))

💾 SAVING CLEANED FULL DATASET

✅ Saved: kompas_cleaned_full.csv
   Rows: 6,834
   Columns: ['date', 'title', 'content', 'content_length']
   Size: 5.83 MB

📋 Sample:


,date,title,content,content_length
0,2022-01-20,"Sebut Transformasi Energi Butuh Dana Besar, Jokowi: Indonesia Minta Kontribusi Negara Maju untuk...","JAKARTA, KOMPAS.com -Presiden Joko Widodo mengatakan, transformasi energi memerlukan dana yang b...",485
1,2022-01-20,Satgas: Kasus Covid-19 dari Berbagai Varian di Indonesia Masih Terkendali,"JAKARTA, KOMPAS.com- Juru Bicara Satuan Tugas (Satgas) Penanganan Covid-19, Wiku Adisasmito meng...",790
2,2022-01-20,"UPDATE 20 Januari: Sebaran 2.116, Kasus Harian Covid-19 di Indonesia, Jakarta Tertinggi","JAKARTA, KOMPAS.com -Pemerintah memperbarui informasi perkembangan kasus harian Covid-19 pada Ka...",381
3,2022-01-20,UPDATE 20 Januari: 12.328 Kasus Aktif Covid-19 di Indonesia,"JAKARTA, KOMPAS.com- Pemerintah menyampaikan terdapat 12.328 kasus aktif Covid-19 pada Kamis (20...",365
4,2022-01-20,Kemenlu: Indonesia Bakal Kedatangan Vaksin dari Jerman dan China,"JAKARTA, KOMPAS.com -Kementerian Luar Negeri (Kemenlu) melaporkan, dalam waktu dekat Indonesia b...",900


## 🎲 Step 8: Random Sampling for Manual Labeling (Seed=42)

In [11]:
print("="*70)
print("🎲 RANDOM SAMPLING FOR MANUAL LABELING")
print("="*70)

if len(df) == 0:
    print("\n⚠️ Skipping sampling (empty dataset)")
else:
    # Set sample size (600-1000)
    # Adjust based on available data
    MAX_SAMPLE = 800
    SAMPLE_SIZE = min(MAX_SAMPLE, len(df_clean))  # Don't sample more than available
    
    print(f"\nAvailable data: {len(df_clean):,} articles")
    print(f"Sample size: {SAMPLE_SIZE}")
    print(f"Random seed: 42 (reproducible)")
    
    if SAMPLE_SIZE < 600:
        print(f"\n⚠️ WARNING: Sample size < 600 (minimum recommended)")
        print(f"   Consider re-scraping more 2019-2024 data")
    
    # Random sampling with seed=42
    df_sample = df_clean.sample(n=SAMPLE_SIZE, random_state=42)
    df_sample = df_sample.sort_values('date').reset_index(drop=True)
    
    print(f"\n✅ Sampled {len(df_sample):,} articles")
    
    # Check distribution
    print(f"\n📊 Sample distribution:")
    sample_monthly = df_sample.groupby(df_sample['date'].dt.to_period('M')).size()
    print(f"   Months covered: {len(sample_monthly)}")
    print(f"   Mean per month: {sample_monthly.mean():.1f}")
    
    print(f"\n📅 Date range:")
    print(f"   From: {df_sample['date'].min().date()}")
    print(f"   To: {df_sample['date'].max().date()}")

🎲 RANDOM SAMPLING FOR MANUAL LABELING

Available data: 6,834 articles
Sample size: 800
Random seed: 42 (reproducible)

✅ Sampled 800 articles

📊 Sample distribution:
   Months covered: 33
   Mean per month: 24.2

📅 Date range:
   From: 2022-01-23
   To: 2024-09-29


## 📝 Step 9: Prepare for Manual Labeling

In [12]:
print("="*70)
print("📝 PREPARING FOR MANUAL LABELING")
print("="*70)

if len(df) == 0 or 'df_sample' not in locals():
    print("\n⚠️ Skipping (no sample data)")
else:
    # Create labeling dataframe
    df_labeling = df_sample.copy()
    
    # Add ID column
    df_labeling.insert(0, 'id', range(1, len(df_labeling) + 1))
    
    # Add text preview (first 300 chars for quick reading)
    df_labeling['text_preview'] = df_labeling['content'].str[:300] + '...'
    
    # Add empty sentiment column (to be filled manually)
    df_labeling['sentiment'] = ''
    
    # Add confidence column (optional)
    df_labeling['confidence'] = ''
    
    # Add notes column (optional)
    df_labeling['notes'] = ''
    
    # Reorder columns for easy labeling
    df_labeling = df_labeling[[
        'id',
        'date',
        'title',
        'text_preview',
        'sentiment',      # ← FILL THIS!
        'confidence',     # Optional
        'notes',          # Optional
        'content',        # Full text (for reference)
        'content_length'
    ]]
    
    print(f"\n✅ Prepared labeling dataset")
    print(f"   Rows: {len(df_labeling):,}")
    print(f"   Columns: {df_labeling.columns.tolist()}")
    
    print(f"\n📋 Sample for labeling:")
    display(df_labeling[['id', 'date', 'title', 'text_preview', 'sentiment']].head(5))

📝 PREPARING FOR MANUAL LABELING

✅ Prepared labeling dataset
   Rows: 800
   Columns: ['id', 'date', 'title', 'text_preview', 'sentiment', 'confidence', 'notes', 'content', 'content_length']

📋 Sample for labeling:


,id,date,title,text_preview,sentiment
0,1,2022-01-23,"Polemik Arteria Dahlan Jadi Pembelajaran Kader, Hasto: Dalam Politik Hati-hati Berbicara","BALI, KOMPAS.com- Sekretaris Jenderal PDI-P Hasto Kristiyanto meminta seluruh pengurus dan kader...",
1,2,2022-01-23,"HUT Ke-75 Megawati, Pramono Anung: Politik Ibu Ketum Jangka Panjang, Tidak Grasa-grusu","BALI, KOMPAS.com- Mantan Sekretaris Jenderal PDI Perjuangan Pramono Anung mengungkapkan bahwa Ke...",
2,3,2022-01-24,"Pengamat Tebak Pesan Anies ke Giring: Kalau Sumbang Suaranya, Jangan Ngomong Politik Dulu Deh...","JAKARTA, KOMPAS.com- Direktur Eksekutif Parameter Politik Indonesia, Adi Prayitno menebak pesan ...",
3,4,2022-01-24,UPDATE: 20.867 Kasus Aktif Covid-19 di Indonesia,"JAKARTA, KOMPAS.com- Pemerintah menyampaikan terdapat 20.867 kasus aktif Covid-19 pada Senin (24...",
4,5,2022-01-25,UPDATE 25 Januari: 7.483 Kasus Suspek Covid-19 di Indonesia,"JAKARTA, KOMPAS.com- Pemerintah mencatat, hingga Selasa (25/1/2022) ada 7.483 suspek Covid-19 di...",


## 💾 Step 10: Save Files for Labeling

In [13]:
print("="*70)
print("💾 SAVING LABELING FILES")
print("="*70)

if len(df) == 0 or 'df_labeling' not in locals():
    print("\n⚠️ Skipping (no labeling data)")
else:
    # 1. Save CSV for labeling
    labeling_file = f'kompas_seed_for_labeling_{SAMPLE_SIZE}.csv'
    df_labeling.to_csv(labeling_file, index=False, encoding='utf-8-sig')
    
    file_size = os.path.getsize(labeling_file) / (1024**2)
    print(f"\n1️⃣ CSV for labeling:")
    print(f"   File: {labeling_file}")
    print(f"   Size: {file_size:.2f} MB")
    print(f"   ℹ️ Open in Excel and fill 'sentiment' column")
    
    # 2. Save Excel with instructions
    excel_file = f'kompas_seed_for_labeling_{SAMPLE_SIZE}.xlsx'
    with pd.ExcelWriter(excel_file, engine='openpyxl') as writer:
        # Main labeling sheet
        df_labeling.to_excel(writer, sheet_name='Labeling', index=False)
        
        # Guidelines sheet
        guidelines = pd.DataFrame({
            'Category': ['POSITIVE', 'NEGATIVE', 'NEUTRAL'],
            'Description': [
                'Berita positif: ekonomi naik, kebijakan sukses, investasi masuk, stabilitas',
                'Berita negatif: krisis, konflik, korupsi, ekonomi turun, demo/protes',
                'Berita netral: pengumuman rutin, laporan, meeting, tidak ada sentimen jelas'
            ],
            'Keywords': [
                'naik, tumbuh, sukses, optimis, bagus, untung, stabil, meningkat',
                'turun, krisis, gagal, konflik, demo, korupsi, menurun, jatuh',
                'umumkan, laporkan, adakan, hadiri, bahas, diskusi, rapat'
            ],
            'Examples': [
                'Ekonomi Indonesia Tumbuh 5.2%, Investasi Asing Meningkat',
                'Rupiah Melemah, Demo Tolak Omnibus Law, Menteri Korupsi',
                'Presiden Hadiri Rapat Kabinet, Menteri Laporkan Kinerja'
            ]
        })
        guidelines.to_excel(writer, sheet_name='Guidelines', index=False)
    
    file_size = os.path.getsize(excel_file) / (1024**2)
    print(f"\n2️⃣ Excel with guidelines:")
    print(f"   File: {excel_file}")
    print(f"   Size: {file_size:.2f} MB")
    print(f"   ℹ️ Open in Excel, see 'Guidelines' sheet for instructions")
    
    print(f"\n{'='*70}")
    print(f"✅ LABELING FILES SAVED!")
    print(f"{'='*70}")

💾 SAVING LABELING FILES

1️⃣ CSV for labeling:
   File: kompas_seed_for_labeling_800.csv
   Size: 0.93 MB
   ℹ️ Open in Excel and fill 'sentiment' column

2️⃣ Excel with guidelines:
   File: kompas_seed_for_labeling_800.xlsx
   Size: 0.29 MB
   ℹ️ Open in Excel, see 'Guidelines' sheet for instructions

✅ LABELING FILES SAVED!


## 📊 Step 11: Save Remaining Unlabeled Data

In [14]:
print("="*70)
print("📊 SAVING REMAINING UNLABELED DATA")
print("="*70)

if len(df) == 0 or 'df_sample' not in locals():
    print("\n⚠️ Skipping (no data)")
else:
    # Get remaining data (not in sample)
    sampled_ids = df_sample.index
    df_remaining = df_clean.loc[~df_clean.index.isin(sampled_ids)].copy()
    df_remaining = df_remaining.reset_index(drop=True)
    
    # Save
    remaining_file = 'kompas_remaining_for_prediction.csv'
    df_remaining.to_csv(remaining_file, index=False, encoding='utf-8-sig')
    
    file_size = os.path.getsize(remaining_file) / (1024**2)
    
    print(f"\n✅ Saved: {remaining_file}")
    print(f"   Rows: {len(df_remaining):,}")
    print(f"   Size: {file_size:.2f} MB")
    print(f"   ℹ️ This will be predicted after training on labeled seed")

📊 SAVING REMAINING UNLABELED DATA

✅ Saved: kompas_remaining_for_prediction.csv
   Rows: 6,034
   Size: 5.21 MB
   ℹ️ This will be predicted after training on labeled seed


## 📋 SUMMARY

In [ ]:
print("="*70)
print("📋 FINAL SUMMARY")
print("="*70)

if len(df) == 0:
    print(f"\n❌ NO DATA IN 2019-2024 RANGE!")
    print(f"\n⚠️ ACTION REQUIRED:")
    print(f"   1. Check original Kompas data - mostly 2025-2026")
    print(f"   2. Re-scrape with date filters: 2019-09-01 to 2024-09-30")
    print(f"   3. Or use only CNBC data for now")
else:
    print(f"\n✅ COMPLETED SUCCESSFULLY!\n")
    
    print(f"📊 Dataset Summary:")
    print(f"   Total cleaned articles: {len(df_clean):,}")
    print(f"   Period: {df_clean['date'].min().date()} to {df_clean['date'].max().date()}")
    
    print(f"\n📁 Files Created:")
    print(f"\n1️⃣ kompas_cleaned_full.csv")
    print(f"   • Full cleaned dataset ({len(df_clean):,} articles)")
    print(f"   • Ready for sentiment analysis & time series")
    print(f"   • Columns: date, title, content, content_length")
    
    if 'df_labeling' in locals():
        print(f"\n2️⃣ kompas_seed_for_labeling_{SAMPLE_SIZE}.csv")
        print(f"   • Random sample ({SAMPLE_SIZE} articles, seed=42)")
        print(f"   • For manual labeling")
        print(f"   • Fill 'sentiment' column: positive/negative/neutral")
        
        print(f"\n3️⃣ kompas_seed_for_labeling_{SAMPLE_SIZE}.xlsx")
        print(f"   • Excel version with guidelines sheet")
        print(f"   • Easier for manual labeling")
        print(f"   • Includes labeling instructions")
        
        print(f"\n4️⃣ kompas_remaining_for_prediction.csv")
        print(f"   • Remaining unlabeled data ({len(df_remaining):,} articles)")
        print(f"   • Will be predicted after training")
    
    print(f"\n{'='*70}")
    print(f"🎯 NEXT STEPS:")
    print(f"{'='*70}")
    if 'df_labeling' in locals():
        print(f"\n1. Open: kompas_seed_for_labeling_{SAMPLE_SIZE}.xlsx")
        print(f"2. Read: Guidelines sheet for labeling instructions")
        print(f"3. Label: Fill 'sentiment' column for {SAMPLE_SIZE} articles")
        print(f"4. Add dropdown: Data → Data Validation → List → positive,negative,neutral")
        print(f"5. Save: As kompas_seed_for_labeling_{SAMPLE_SIZE}_LABELED.xlsx")
        print(f"6. Merge: Combine with CNBC labeled data")
        print(f"7. Train: Use combined data to train IndoBERT")
        print(f"8. Predict: Apply model to remaining articles")
    
    print(f"\n{'='*70}")
    print(f"✨ Random seed: 42 (reproducible sampling)")
    print(f"✨ Ready for sentiment analysis!")
    print(f"{'='*70}")